# In-Class Assignment — Python for Feature Extraction
**Time:** 20 minutes  |  **Points:** 20


....


Dataset file: `product_reviews.txt`

## Load the dataset
Download it from **Canvas**, then run the upload cell below to select the file from your computer.


In [6]:
from google.colab import files

uploaded = files.upload()
filename = next(iter(uploaded))

Saving product_reviews.txt to product_reviews (1).txt


## Q1 (2 point) — Load & Read & inspect

## Note about the header
**Important:** The dataset file includes a **header row** (column names).  
Make sure the header is used as the column names — it **should NOT appear as a data row** in your DataFrame.

Tip: Use the filename variable printed above when reading the file.
After Reading, check that your columns are `id` and `text`.
Print: **(a)** `df.shape`, **(b)** `df.head(3\)`.  


In [7]:
import pandas as pd

df = pd.read_csv(filename, sep="\t")

print(df.head())
print(df.columns)

   id                                               text
0   1  Love this blender! Smoothies are super creamy ...
1   2  Terrible quality... stopped working after 2 da...
2   3      Good value for the price. Shipping was quick.
3   4  Not as described. Missing parts and the box wa...
4   5  Customer support helped me get a replacement. ...
Index(['id', 'text'], dtype='object')


## Q2 (4 points) — Basic handcrafted features  
Create these columns and then display the DataFrame:
- `word_count` = number of words  
- `char_count` = number of characters  
- `avg_word_len` = average word length (ignore punctuation)  
- `excl_count` = number of `!` characters  

Print: `id, word_count, char_count, avg_word_len, excl_count`.


In [8]:
import string

df['word_count'] = df['text'].apply(lambda x: len(str(x).split()))
df['char_count'] = df['text'].apply(lambda x: len(str(x)))

def avg_word_len(text):
    words = str(text).translate(str.maketrans('', '', string.punctuation)).split()
    return sum(len(word) for word in words) / len(words) if words else 0

df['avg_word_len'] = df['text'].apply(avg_word_len)

df['excl_count'] = df['text'].apply(lambda x: str(x).count('!'))

print(df[['id', 'word_count', 'char_count', 'avg_word_len', 'excl_count']])

   id  word_count  char_count  avg_word_len  excl_count
0   1           9          55      5.000000           1
1   2           7          51      5.571429           3
2   3           8          45      4.500000           0
3   4          10          56      4.500000           0
4   5           8          53      5.500000           1
5   6          10          51      4.000000           0
6   7           9          50      4.444444           0
7   8           6          40      5.500000           0
8   9           7          52      6.285714           0
9  10           7          48      5.571429           0


## Q3 (6 points) — Bag-of-Words (CountVectorizer)  
Use `CountVectorizer(stop_words="english")` on `df["text"]`. Print:
1) vocabulary size (number of features)  
2) top 10 words by **total count** across all documents (word + count)


In [9]:
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

vectorizer = CountVectorizer(stop_words='english')
X = vectorizer.fit_transform(df['text'])

print("Vocabulary size:", len(vectorizer.vocabulary_))

word_counts = np.array(X.sum(axis=0)).flatten()
words = vectorizer.get_feature_names_out()

top_indices = word_counts.argsort()[-10:][::-1]

print("\nTop 10 words:")
for i in top_indices:
    print(words[i], word_counts[i])

Vocabulary size: 50

Top 10 words:
wouldn 1
works 1
working 1
want 1
value 1
twice 1
thanks 1
terrible 1
support 1
super 1


## Q4 (4 points) — Bigram features  
Use `CountVectorizer(stop_words="english", ngram_range=(2,2))`.  
Print the top 5 bigrams by total count (bigram + count).


In [10]:
vectorizer_bigram = CountVectorizer(stop_words='english', ngram_range=(2,2))
X_bigram = vectorizer_bigram.fit_transform(df['text'])

bigram_counts = np.array(X_bigram.sum(axis=0)).flatten()
bigrams = vectorizer_bigram.get_feature_names_out()

top_indices = bigram_counts.argsort()[-5:][::-1]

print("\nTop 5 bigrams:")
for i in top_indices:
    print(bigrams[i], bigram_counts[i])


Top 5 bigrams:
wouldn buy 1
works fine 1
working days 1
want refund 1
value price 1


## Q5 (4 points) — TF-IDF features  
Use `TfidfVectorizer(stop_words="english", ngram_range=(1,2))`.  
Compute the **average TF-IDF** score of each term across documents and print the top 5 terms (term + avg score).


In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words='english', ngram_range=(1,2))
X_tfidf = tfidf.fit_transform(df['text'])

avg_tfidf = np.mean(X_tfidf.toarray(), axis=0)
terms = tfidf.get_feature_names_out()

top_indices = avg_tfidf.argsort()[-5:][::-1]

print("\nTop 5 TF-IDF terms:")
for i in top_indices:
    print(terms[i], avg_tfidf[i])


Top 5 TF-IDF terms:
setup 0.03779644730092272
setup easy 0.03779644730092272
instructions 0.03779644730092272
easy 0.03779644730092272
clear 0.03779644730092272


## Grading Checklist
- Q1: correct prints  
- Q2: correct feature columns + requested display  
- Q3: correct vocabulary size + correct top 10 words by total count  
- Q4: correct top 5 bigrams by total count  
- Q5: correct top 5 TF-IDF terms by average score
